In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp 

CATALOG = spark.conf.get("catalog")
BRONZE_SCHEMA = spark.conf.get("bronze_schema")

BASE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw"
LANDING_PATH = f"{BASE_PATH}/landing"
REFERENCE_FILE = f"{BASE_PATH}/reference/ratings.csv"
VIEWS_FILE = f"{BASE_PATH}/reference/netflix_views.csv"

In [0]:
@dp.table(name = f"{BRONZE_SCHEMA}.bronze_netflix")
def bronze_table():
    return(
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("rescuedDataColumn", "_rescued_data")
            .option("header", "true")
            .option("inferSchema", "false")
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .load(LANDING_PATH)
            .withColumn("ingestion_time", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path"))
    )

In [0]:
@dp.table(name=f"{BRONZE_SCHEMA}.rating_reference")
def rating_reference():
    return (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("sep", ";")
            .csv(REFERENCE_FILE)
    )

In [0]:
@dp.table(name=f"{BRONZE_SCHEMA}.views_events")
def views_events():
    return (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(VIEWS_FILE)
    )